In [0]:
import os
import sys
from uuid import uuid4

import boto3
import pandas as pd

from arcgis.geocoding import geocode
from arcgis.geometry import Geometry, MultiPoint
from arcgis.geometry.filters import intersects
from arcgis.gis import GIS, ItemProperties, ItemTypeEnum

from arcgis.raster import utils as raster_utils
from arcgis.raster import ImageryLayer
from arcgis.raster.functions import band_arithmetic

In [0]:
# If you are using a guest account for the hackathon you can connect to the
# CSU ArcGIS Online org it is as simple as passing in the assigned guest
# username and password:
gis = GIS('https://csurams.maps.arcgis.com', 'csuguest10', 'CSUguest10!')

# prove I'm connected
me = gis.users.me
print(me.username)

csuguest10


In [0]:
# use get() when you know the ID of an existing item

# Grab the USDA NAIP Color Infrared Imagery from the Living Atlas
international_segment_feature_layer = gis.content.get('ffc2e851a5fd4ef7a1e2581072b5c46d')
print(f'{international_segment_feature_layer.title}, is a {international_segment_feature_layer.type}')

PPQ International Segments Feature Layer, is a Feature Service


In [0]:
# Get the actual layer from the item
layer = international_segment_feature_layer.layers[0]

# Pull data into a pandas dataframe
international_df = layer.query(
    where="1=1",
    out_fields="*",
    as_df=True
)

# See how much data you have
print(f"Total records: {len(international_df)}")
print(f"Columns available: {international_df.columns.tolist()}")

# Preview the data
display(international_df.head(10))

Total records: 544082
Columns available: ['OBJECTID', 'DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'PAYLOAD', 'SEATS', 'PASSENGERS', 'FREIGHT', 'MAIL', 'DISTANCE', 'RAMP_TO_RAMP', 'AIR_TIME', 'UNIQUE_CARRIER', 'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'UNIQUE_CARRIER_ENTITY', 'REGION', 'CARRIER', 'CARRIER_NAME', 'CARRIER_GROUP', 'CARRIER_GROUP_NEW', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_COUNTRY', 'ORIGIN_COUNTRY_NAME', 'ORIGIN_WAC', 'ORIGIN_LAT', 'ORIGIN_LON', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_COUNTRY', 'DEST_COUNTRY_NAME', 'DEST_WAC', 'DEST_LAT', 'DEST_LON', 'AIRCRAFT_GROUP', 'AIRCRAFT_TYPE', 'AIRCRAFT_CONFIG', 'YEAR_WADS', 'QUARTER_WADS', 'MONTH_WADS', 'DISTANCE_GROUP', 'CLASS', 'SHAPE']


,OBJECTID,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,UNIQUE_CARRIER,AIRLINE_ID,UNIQUE_CARRIER_NAME,UNIQUE_CARRIER_ENTITY,REGION,CARRIER,CARRIER_NAME,CARRIER_GROUP,CARRIER_GROUP_NEW,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_COUNTRY,ORIGIN_COUNTRY_NAME,ORIGIN_WAC,ORIGIN_LAT,ORIGIN_LON,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_COUNTRY,DEST_COUNTRY_NAME,DEST_WAC,DEST_LAT,DEST_LON,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR_WADS,QUARTER_WADS,MONTH_WADS,DISTANCE_GROUP,CLASS,SHAPE
0,401,13,13,534000,2094,1439,0,0,1582,3056,2716,B6,20409,JetBlue Airways,16673,L,B6,JetBlue Airways,3,3,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,15013,1501302,35013,STI,"Santiago, Dominican Republic",DO,Dominican Republic,224,19.45404,-70.70745,6,694,1,2022,1,2,4,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
1,402,15,14,726600,2800,2121,0,0,1582,3135,2759,B6,20409,JetBlue Airways,16673,L,B6,JetBlue Airways,3,3,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,15013,1501302,35013,STI,"Santiago, Dominican Republic",DO,Dominican Republic,224,19.45404,-70.70745,6,699,1,2022,1,2,4,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
2,403,0,23,1169315,4282,2810,440,0,2395,0,0,S4,20360,Sata Internacional,9469D,I,S4,Sata Internacional,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,699,1,2020,1,1,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
3,404,0,20,1016796,3728,2762,917,0,2395,0,0,S4,20360,Sata Internacional,9469D,I,S4,Sata Internacional,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,699,1,2020,1,2,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
4,405,0,13,660917,2418,1328,156,0,2395,0,0,S4,20360,Sata Internacional,9469D,I,S4,Sata Internacional,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,699,1,2020,1,3,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
5,406,0,5,228800,840,278,0,0,2395,0,0,TP,19576,Tap-Portuguese Airlines,9469A,I,TP,Tap-Portuguese Airlines,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,824,1,2020,3,7,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
6,407,0,12,610078,2236,865,1109,0,2395,0,0,S4,20360,Sata Internacional,9469D,I,S4,Sata Internacional,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,699,1,2020,3,7,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
7,408,0,13,660917,2426,510,180,0,2395,0,0,S4,20360,Sata Internacional,9469D,I,S4,Sata Internacional,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,699,1,2020,3,8,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
8,409,0,13,594880,2182,1180,0,0,2395,0,0,TP,19576,Tap-Portuguese Airlines,9469A,I,TP,Tap-Portuguese Airlines,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta Delgada, Portugal",PT,Portugal,469,37.73962,-25.66855,6,824,1,2020,3,8,5,F,"{""x"": 1909432.8728, ""y"": 531173.3172999993, ""s..."
9,410,0,13,594880,2184,1168,0,0,2395,0,0,TP,19576,Tap-Portuguese Airlines,9469A,I,TP,Tap-Portuguese Airlines,0,0,10721,1072102,30721,BOS,"Boston, MA",US,United States,13,42.35866,-71.05674,14051,1405107,34051,PDL,"Ponta

In [0]:
# Search for fruit fly related content
results = gis.content.search(
    query='fruit fly PPQ',
    max_items=20,
    outside_org=True
)

for item in results:
    print(f"{item.title} | {item.type} | ID: {item.id}")

PPQ Fruit Fly Image | Image | ID: 4852c12beb9046d7add62d275d901fdb
PPQ TX Quarantine Areas Feature Layer | Feature Service | ID: b0b5666860f34ab2b4a27f4dff35597b
PPQ Fruit Fly Detections Summary Feature Layer | Feature Service | ID: 24ed5194234f4d0a824430b745b2b8f4
PPQ MEDFF Small Image | Image | ID: ae8737ba5ea047aa9f196222288c24ee


In [0]:
# Replace ID_HERE with what comes back from search
domestic_segments = gis.content.get('f2d35d3da8644c0ca4b7b6ac2c1486f6')
print(f'{domestic_segments.title}, is a {domestic_segments.type}')

PPQ Domestic Segments Feature Layer, is a Feature Service


In [0]:
domestic_layer = domestic_segments.layers[0]

# domestic_df = domestic_layer.query(
#     where="1=1",
#     out_fields="*",
#     as_df=True
# )

# print(f"Total records: {len(domestic_df)}")
# print(f"Columns: {domestic_df.columns.tolist()}")
# display(domestic_df.head(10))

# Just get 100 rows to see structure
# domestic_df = domestic_layer.query(
#     where="1=1",
#     out_fields="*",
#     as_df=True,
#     result_record_count=100
# )

# Filter WHILE loading from ArcGIS
# Much more efficient!
domestic_df = domestic_layer.query(
    where="DEST_STATE_NM IN ('California', 'Texas', 'Florida')",
    out_fields="*",
    as_df=True
)

print(f"Total filtered records: {len(domestic_df)}")
display(domestic_df.head(10))

# print(f"Columns: {domestic_df.columns.tolist()}")
# display(domestic_df.head(5))

Total filtered records: 608918


,OBJECTID,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,UNIQUE_CARRIER,AIRLINE_ID,UNIQUE_CARRIER_NAME,UNIQUE_CARRIER_ENTITY,REGION,CARRIER,CARRIER_NAME,CARRIER_GROUP,CARRIER_GROUP_NEW,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,ORIGIN_LAT,ORIGIN_LON,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_FIPS,DEST_STATE_NM,DEST_WAC,DEST_LAT,DEST_LON,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR_BTS,QUARTER_BTS,MONTH_BTS,DISTANCE_GROUP,CLASS,SHAPE
0,396,0,1,280443,0,0,24868,0,3993,500,468,5Y,20007,Atlas Air Inc.,6087,D,5Y,Atlas Air Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,11697,1169706,32467,FLL,"Fort Lauderdale, FL",FL,12,Florida,33,26.12401,-80.14357,8,819,2,2024,3,8,8,P,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
1,507,30,29,1098643,4814,3387,1522,0,3266,11885,11140,UA,19977,United Air Lines Inc.,0A875,D,UA,United Air Lines Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,838,1,2024,2,6,7,F,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
2,508,31,28,1077046,4648,3855,0,0,3266,11277,10583,UA,19977,United Air Lines Inc.,0A875,D,UA,United Air Lines Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,838,1,2024,3,7,7,F,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
3,509,1,1,42710,179,162,0,0,3266,388,366,UA,19977,United Air Lines Inc.,0A875,D,UA,United Air Lines Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,839,1,2024,3,8,7,F,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
4,510,30,27,1068361,4482,3706,0,0,3266,10935,10277,UA,19977,United Air Lines Inc.,0A875,D,UA,United Air Lines Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,838,1,2024,3,8,7,F,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
5,511,25,25,963871,4150,3102,0,0,3266,10155,9507,UA,19977,United Air Lines Inc.,0A875,D,UA,United Air Lines Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,838,1,2024,3,9,7,F,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
6,512,0,1,224872,0,0,100456,0,3266,478,451,5Y,20007,Atlas Air Inc.,6087,D,5Y,Atlas Air Inc.,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,627,1,2024,4,10,7,P,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
7,513,0,1,480900,0,0,236527,0,3266,442,417,KAQ,20370,Kalitta Air LLC,6683,D,KAQ,Kalitta Air LLC,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,8,819,2,2024,4,10,7,P,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
8,514,0,1,232018,0,0,207713,0,3266,458,431,FX,20107,Federal Express Corporation,6200,D,FX,Federal Express Corporation,3,3,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,6,683,2,2024,4,10,7,P,"{""x"": -2773630.3121000007, ""y"": 3274769.583900..."
9,515,0,1,264000,0,0,112620,0,3266,0,0,ADB,20110,Antonov Company,9488B,I,ADB,Antonov Company,0,0,10299,1029906,30299,ANC,"Anchorage, AK",AK,2,Alaska,1,61.21753,-149.85815,12266,1226603,31453,IAH,"Houston, TX",TX,48,Texas,74,29.76078,-95.36952,8,880,2,2024,4,10,7,P,"{""x"": -2773630.3121000007, ""y"": 3274769.5

In [0]:
# Replace ID_HERE with what comes back from search
fruit_fly_detection_summary = gis.content.get('24ed5194234f4d0a824430b745b2b8f4')
print(f'{fruit_fly_detection_summary.title}, is a {fruit_fly_detection_summary.type}')

PPQ Fruit Fly Detections Summary Feature Layer, is a Feature Service


In [0]:
fruit_fly_layer = fruit_fly_detection_summary.layers[0]

fruit_fly_df = fruit_fly_layer.query(
    where="1=1",
    out_fields="*",
    as_df=True
)

print(f"Total records: {len(fruit_fly_df)}")
print(f"Columns: {fruit_fly_df.columns.tolist()}")
display(fruit_fly_df.head(20))


Total records: 305
Columns: ['OBJECTID', 'NAME', 'STATE_NAME', 'monthyear', 'CommonName', 'Count_', 'Shape__Area', 'Shape__Length', 'SHAPE']


,OBJECTID,NAME,STATE_NAME,monthyear,CommonName,Count_,Shape__Area,Shape__Length,SHAPE
0,1,Cameron County,Texas,01/2025,Mexican Fruit Fly,13.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
1,2,Cameron County,Texas,02/2022,Mexican Fruit Fly,10.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
2,3,Cameron County,Texas,02/2024,Mexican Fruit Fly,2.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
3,4,Cameron County,Texas,02/2025,Mexican Fruit Fly,14.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
4,5,Cameron County,Texas,03/2022,Mexican Fruit Fly,14.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
5,6,Cameron County,Texas,03/2024,Mexican Fruit Fly,13.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
6,7,Cameron County,Texas,03/2025,Mexican Fruit Fly,4.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
7,8,Cameron County,Texas,04/2022,Mexican Fruit Fly,11.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
8,9,Cameron County,Texas,04/2024,Mexican Fruit Fly,5.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
9,10,Cameron County,Texas,04/2025,Mexican Fruit Fly,26.0,2439115116.406128,939031.574311,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."


In [0]:
df = spark.table("`hackathon-resources-shared`.information_schema.tables")
display(df)



table_catalog,table_schema,table_name,table_type,is_insertable_into,commit_action,table_owner,comment,created,created_by,last_altered,last_altered_by,data_source_format,storage_sub_directory,storage_path
hackathon-resources-shared,information_schema,schema_privileges,VIEW,NO,PRESERVE,System user,Lists principals that have privileges on the schemas in the catalog.,2026-04-24T19:26:45.102Z,System user,2026-04-24T19:37:19.357Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,table_privileges,VIEW,NO,PRESERVE,System user,Lists principals that have privileges on the tables and views in the catalog.,2026-04-24T19:26:45.227Z,System user,2026-04-24T19:37:19.439Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,routine_tags,VIEW,NO,PRESERVE,System user,null,2026-04-24T19:26:45.478Z,System user,2026-04-24T19:37:19.665Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,volume_privileges,VIEW,NO,PRESERVE,System user,null,2026-04-24T19:26:45.355Z,System user,2026-04-24T19:37:19.575Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,tables,VIEW,NO,PRESERVE,System user,Describes tables and views defined within the catalog.,2026-04-24T19:26:45.248Z,System user,2026-04-24T19:37:19.452Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,views,VIEW,NO,PRESERVE,System user,Describes view specific information about the views in the catalog.,2026-04-24T19:26:45.270Z,System user,2026-04-24T19:37:19.466Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,referential_constraints,VIEW,NO,PRESERVE,System user,Describes referential (foreign key) constraints defined in the catalog.,2026-04-24T19:26:44.986Z,System user,2026-04-24T19:37:19.288Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,parameters,VIEW,NO,PRESERVE,System user,Describes parameters of routines (functions) in the catalog.,2026-04-24T19:26:45.057Z,System user,2026-04-24T19:37:19.328Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,column_tags,VIEW,NO,PRESERVE,System user,null,2026-04-24T19:26:45.436Z,System user,2026-04-24T19:37:19.634Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null
hackathon-resources-shared,information_schema,catalog_privileges,VIEW,NO,PRESERVE,System user,Lists principals that have privileges on the catalogs.,2026-04-24T19:26:44.856Z,System user,2026-04-24T19:37:19.201Z,System user,UNKNOWN_DATA_SOURCE_FORMAT,null,null


In [0]:
spark.createDataFrame(fruit_fly_df)\
     .write.mode("overwrite")\
     .saveAsTable("fruit_fly_detections")

In [0]:
detections_df = spark.table(
    "fruit_fly_detections"
).toPandas()

print(f"Total: {len(detections_df)}")
display(detections_df.head(10))

Total: 305


,OBJECTID,NAME,STATE_NAME,monthyear,CommonName,Count_,Shape__Area,Shape__Length,SHAPE
0,1,Cameron County,Texas,01/2025,Mexican Fruit Fly,13.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
1,2,Cameron County,Texas,02/2022,Mexican Fruit Fly,10.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
2,3,Cameron County,Texas,02/2024,Mexican Fruit Fly,2.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
3,4,Cameron County,Texas,02/2025,Mexican Fruit Fly,14.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
4,5,Cameron County,Texas,03/2022,Mexican Fruit Fly,14.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
5,6,Cameron County,Texas,03/2024,Mexican Fruit Fly,13.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
6,7,Cameron County,Texas,03/2025,Mexican Fruit Fly,4.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
7,8,Cameron County,Texas,04/2022,Mexican Fruit Fly,11.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
8,9,Cameron County,Texas,04/2024,Mexican Fruit Fly,5.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...
9,10,Cameron County,Texas,04/2025,Mexican Fruit Fly,26.0,2.439115e+09,939031.574311,b'\x01\x06\x00\x00\x00\'\x00\x00\x00\x01\x03\x...


In [0]:
species = detections_df.groupby(
    'CommonName'
)['Count_'].sum().reset_index()

species.columns = ['Species', 'Total_Detections']
species = species.sort_values(
    'Total_Detections',
    ascending=False
)

display(species)

,Species,Total_Detections
5,Oriental Fruit Fly,1037.0
4,Mexican Fruit Fly,723.0
2,Mediterranean Fruit Fly,205.0
9,Zeugodacus Tau,135.0
6,Peach Fruit Fly,36.0
1,Guava Fruit Fly,20.0
0,Caribbean Fruit Fly,8.0
7,Queensland Fruit Fly,3.0
3,Melon Fruit Fly,2.0
8,Sapote Fruit Fly,1.0


In [0]:
states = detections_df.groupby(
    'STATE_NAME'
)['Count_'].sum().reset_index()

states.columns = ['State', 'Total_Detections']
states = states.sort_values(
    'Total_Detections',
    ascending=False
)

display(states)

,State,Total_Detections
0,California,1457.0
2,Texas,701.0
1,Florida,12.0


In [0]:
seasonal = detections_df.groupby(
    'monthyear'
)['Count_'].sum().reset_index()

seasonal.columns = ['Month_Year', 'Total_Detections']
seasonal = seasonal.sort_values(
    'Month_Year'
)

display(seasonal)

,Month_Year,Total_Detections
0,01/2020,1.0
1,01/2022,1.0
2,01/2023,1.0
3,01/2024,18.0
4,01/2025,24.0
...,...,...
61,12/2021,1.0
62,12/2022,1.0
63,12/2023,96.0
64,12/2024,5.0


In [0]:
# # Cities in top detection states
# # California, Texas, Florida
# high_risk_cities = [
#     # California
#     'Los Angeles', 'San Francisco', 
#     'San Diego', 'Sacramento',
#     # Texas
#     'Houston', 'Dallas', 
#     'San Antonio', 'Austin',
#     # Florida
#     'Miami', 'Orlando', 
#     'Tampa', 'Fort Lauderdale'
# ]

# # Find which countries send most
# # passengers to these cities
# risky_flights = international_df[
#     international_df['DEST_CITY_NAME'].isin(
#         high_risk_cities
#     )
# ].groupby(
#     ['ORIGIN_COUNTRY_NAME', 'DEST_CITY_NAME']
# )['PASSENGERS'].sum().reset_index()

# risky_flights = risky_flights.sort_values(
#     'PASSENGERS',
#     ascending=False
# ).head(20)

# display(risky_flights)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
import numpy as np

# Basic statistics on detection counts
print("Detection Statistics:")
print(f"Total Detections: {np.sum(detections_df['Count_'])}")
print(f"Average per Record: {np.mean(detections_df['Count_']):.2f}")
print(f"Max Single Detection: {np.max(detections_df['Count_'])}")
print(f"Min Single Detection: {np.min(detections_df['Count_'])}")
print(f"Standard Deviation: {np.std(detections_df['Count_']):.2f}")

Detection Statistics:
Total Detections: 2170.0
Average per Record: 7.11
Max Single Detection: 321.0
Min Single Detection: 1.0
Standard Deviation: 26.77


In [0]:
# Basic flight statistics
print("Domestic Flight Statistics:")
print(f"Total Passengers: {np.sum(domestic_df['PASSENGERS']):,}")
print(f"Average Passengers per Route: {np.mean(domestic_df['PASSENGERS']):.2f}")
print(f"Max Passengers on Route: {np.max(domestic_df['PASSENGERS']):,}")

Domestic Flight Statistics:
Total Passengers: 1,361,185,215
Average Passengers per Route: 2235.42
Max Passengers on Route: 78,753


In [0]:
# Get the states where fruit fly invasions occurred
invasion_states = fruit_fly_df['STATE_NAME'].unique().tolist()
print(f"Fruit fly invasion states: {invasion_states}")

# Filter domestic flights to only those landing in invasion states
flights_in_invasion_states = domestic_df[
    domestic_df['DEST_STATE_NM'].isin(invasion_states)
]
print(f"\nFlights landing in invasion states: {len(flights_in_invasion_states):,}")

# Aggregate by destination city: total passengers + keep lat/lon
dest_cities = flights_in_invasion_states.groupby(
    ['DEST_CITY_NAME', 'DEST_STATE_NM']
).agg({
    'PASSENGERS': 'sum',
    'DEST_LAT': 'first',
    'DEST_LON': 'first'
}).reset_index()

dest_cities.columns = ['City', 'State', 'TotalPassengers', 'Lat', 'Lon']
dest_cities = dest_cities.sort_values('TotalPassengers', ascending=False)

print(f"Unique destination cities in invasion states: {len(dest_cities)}")
display(dest_cities.head(20))

Fruit fly invasion states: ['Texas', 'Florida', 'California']

Flights landing in invasion states: 608,918
Unique destination cities in invasion states: 250


,City,State,TotalPassengers,Lat,Lon
53,"Dallas/Fort Worth, TX",Texas,180626165,32.88985,-97.03983
131,"Los Angeles, CA",California,132820928,34.05357,-118.24545
166,"Orlando, FL",Florida,124025956,28.53823,-81.37739
104,"Houston, TX",Texas,121371385,29.76078,-95.36952
199,"San Francisco, CA",California,85944509,37.77712,-122.41966
145,"Miami, FL",Florida,75260628,25.77481,-80.19773
73,"Fort Lauderdale, FL",Florida,70412887,26.12401,-80.14357
198,"San Diego, CA",California,58984665,32.71568,-117.16171
220,"Tampa, FL",Florida,57823454,27.94653,-82.45927
10,"Austin, TX",Texas,51009218,30.26759,-97.74299


In [0]:
%pip install shapely --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from arcgis.features import FeatureLayer
from shapely.geometry import Point as ShapelyPoint, shape
import json

# Use Esri Living Atlas USA Counties (Generalized) service
counties_url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Counties_Generalized_Boundaries/FeatureServer/0"
counties_layer = FeatureLayer(counties_url)

# State FIPS codes for our invasion states
state_fips = {'California': '06', 'Texas': '48', 'Florida': '12'}
state_fips_list = "','".join(state_fips.values())

# Query all county boundaries for the 3 invasion states
print("Querying county boundaries for CA, TX, FL...")
counties_fs = counties_layer.query(
    where=f"STATE_FIPS IN ('{state_fips_list}')",
    out_fields="NAME,STATE_NAME,STATE_FIPS,FIPS",
    return_geometry=True
)

print(f"Retrieved {len(counties_fs.features)} counties")

# Build shapely polygons for point-in-polygon matching
county_polygons = []
for feat in counties_fs.features:
    try:
        geom = feat.geometry
        # Convert ArcGIS geometry to shapely
        if 'rings' in geom:
            from shapely.geometry import Polygon, MultiPolygon
            polys = []
            for ring in geom['rings']:
                polys.append(Polygon(ring))
            if len(polys) == 1:
                shp = polys[0]
            else:
                shp = MultiPolygon([(p.exterior.coords, []) for p in polys])
        else:
            shp = shape(geom)
        
        county_polygons.append({
            'name': feat.attributes.get('NAME', ''),
            'state': feat.attributes.get('STATE_NAME', ''),
            'fips': feat.attributes.get('FIPS', ''),
            'geometry': shp,
            'arcgis_geom': geom
        })
    except Exception as e:
        pass

print(f"Built {len(county_polygons)} county polygon objects")
print(f"Sample: {county_polygons[0]['name']}, {county_polygons[0]['state']}")

Querying county boundaries for CA, TX, FL...
Retrieved 379 counties
Built 379 county polygon objects
Sample: Alameda County, California


In [0]:
# Match each destination city to a county using point-in-polygon
matched_counties = {}  # fips -> county info + passenger total
unmatched = []

for _, city in dest_cities.iterrows():
    if pd.isna(city['Lat']) or pd.isna(city['Lon']):
        unmatched.append(city['City'])
        continue
    
    pt = ShapelyPoint(city['Lon'], city['Lat'])
    found = False
    
    for county in county_polygons:
        try:
            if county['geometry'].contains(pt):
                fips = county['fips']
                if fips in matched_counties:
                    matched_counties[fips]['TotalPassengers'] += city['TotalPassengers']
                    matched_counties[fips]['cities'].append(city['City'])
                else:
                    matched_counties[fips] = {
                        'name': county['name'],
                        'state': county['state'],
                        'fips': fips,
                        'TotalPassengers': city['TotalPassengers'],
                        'cities': [city['City']],
                        'arcgis_geom': county['arcgis_geom']
                    }
                found = True
                break
        except:
            continue
    
    if not found:
        unmatched.append(city['City'])

print(f"Matched {len(matched_counties)} unique counties")
print(f"Unmatched cities: {len(unmatched)}")
if unmatched[:10]:
    print(f"Sample unmatched: {unmatched[:10]}")

# Show top counties by passengers
matched_list = sorted(matched_counties.values(), key=lambda x: x['TotalPassengers'], reverse=True)
for c in matched_list[:15]:
    print(f"  {c['name']}, {c['state']}: {c['TotalPassengers']:,} passengers ({', '.join(c['cities'][:3])})")

Matched 144 unique counties
Unmatched cities: 27
Sample unmatched: ['West Palm Beach/Palm Beach, FL', 'El Paso, TX', 'Panama City, FL', 'Key West, FL', 'Fort Jefferson, FL', 'Crescent City, CA', 'Destin, FL', 'Marathon, FL', 'Mary Esther, FL', 'Marco Island, FL']
  Tarrant County, Texas: 180,627,103 passengers (Dallas/Fort Worth, TX, Arlington, TX, Fort Worth, TX)
  Los Angeles County, California: 158,288,253 passengers (Los Angeles, CA, Burbank, CA, Long Beach, CA)
  Orange County, Florida: 124,025,956 passengers (Orlando, FL)
  Harris County, Texas: 121,371,385 passengers (Houston, TX)
  San Francisco County, California: 85,944,509 passengers (San Francisco, CA)
  Miami-Dade County, Florida: 75,260,674 passengers (Miami, FL, Homestead, FL)
  Broward County, Florida: 70,412,914 passengers (Fort Lauderdale, FL, Pompano Beach, FL)
  San Diego County, California: 59,078,354 passengers (San Diego, CA, Carlsbad, CA, Ramona, CA)
  Hillsborough County, Florida: 57,823,454 passengers (Tampa, 

In [0]:
from arcgis.features import Feature, FeatureLayer
from arcgis.geometry import Polygon
import json

# Re-query counties in WGS84 (lat/lon) so polygons display correctly
print("Re-querying county boundaries in WGS84...")
counties_url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Counties_Generalized_Boundaries/FeatureServer/0"
counties_layer_wgs = FeatureLayer(counties_url)

state_fips_list = "','".join(['06', '48', '12'])

counties_wgs84 = counties_layer_wgs.query(
    where=f"STATE_FIPS IN ('{state_fips_list}')",
    out_fields="NAME,STATE_NAME,FIPS",
    return_geometry=True,
    out_sr=4326
)

print(f"Retrieved {len(counties_wgs84.features)} counties in WGS84")

# Build lookup by FIPS
county_wgs84_map = {}
for feat in counties_wgs84.features:
    fips = feat.attributes.get('FIPS', '')
    county_wgs84_map[fips] = feat

# Build polygon features for matched counties only
polygon_features = []
for fips, info in matched_counties.items():
    if fips in county_wgs84_map:
        county_feat = county_wgs84_map[fips]
        geom = county_feat.geometry
        geom['spatialReference'] = {'wkid': 4326}
        
        attr = {
            'CountyName': info['name'],
            'State': info['state'],
            'FIPS': fips,
            'TotalPassengers': int(info['TotalPassengers']),
            'Cities': ', '.join(info['cities'][:5]),
            'DataType': 'Flight Destination'
        }
        polygon_features.append(Feature(geometry=geom, attributes=attr))

print(f"Built {len(polygon_features)} polygon features for publishing")

Re-querying county boundaries in WGS84...
Retrieved 379 counties in WGS84
Built 144 polygon features for publishing


In [0]:
import json

# Convert to GeoJSON for publishing
geojson_features = []
for feat in polygon_features:
    geom = feat.geometry
    props = feat.attributes
    
    if 'rings' in geom:
        geojson_geom = {
            "type": "Polygon",
            "coordinates": geom['rings']
        }
    else:
        geojson_geom = geom
    
    geojson_features.append({
        "type": "Feature",
        "geometry": geojson_geom,
        "properties": props
    })

geojson_collection = {
    "type": "FeatureCollection",
    "features": geojson_features
}

geojson_path = "/tmp/flight_counties.geojson"
with open(geojson_path, 'w') as f:
    json.dump(geojson_collection, f)

print(f"GeoJSON saved: {len(geojson_features)} county polygons")

# Publish to CSU RAMS ArcGIS
item_props = {
    "title": "Flight Destination Counties - Blue Fill",
    "tags": "flights, counties, hackathon, fruit fly",
    "description": "Counties in CA/TX/FL where inbound flights land. Blue filled polygons showing flight destination coverage in fruit fly invasion states.",
    "type": "GeoJson"
}

# Clean up existing
existing = gis.content.search(query="title:'Flight Destination Counties - Blue Fill' owner:csuguest10", max_items=10)
for item in existing:
    print(f"Deleting existing: {item.title} ({item.id})")
    item.delete()

geojson_item = gis.content.add(item_props, data=geojson_path)
print(f"Uploaded: {geojson_item.title} (ID: {geojson_item.id})")

flight_published = geojson_item.publish()
print(f"Published: {flight_published.title} (ID: {flight_published.id})")
print(f"URL: {flight_published.url}")

GeoJSON saved: 144 county polygons


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Uploaded: Flight Destination Counties - Blue Fill (ID: 2d90381b33b044fea2b03395b7fe0d1f)
Published: Flight Destination Counties - Blue Fill (ID: 9dde783781b044ba9560944774a1ed01)
URL: https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Flight_Destination_Counties_Blue_Fill/FeatureServer


In [0]:
# Apply blue semi-transparent fill renderer
flight_fl = flight_published.layers[0]

blue_renderer = {
    "type": "simple",
    "symbol": {
        "type": "esriSFS",
        "style": "esriSFSSolid",
        "color": [0, 100, 200, 120],  # Blue, semi-transparent
        "outline": {
            "type": "esriSLS",
            "style": "esriSLSSolid",
            "color": [0, 60, 150, 200],  # Darker blue outline
            "width": 1.5
        }
    },
    "label": "Flight Destination County"
}

update_result = flight_fl.manager.update_definition({
    "drawingInfo": {"renderer": blue_renderer}
})
print(f"Renderer update: {update_result}")

flight_published.share(everyone=True)
print("Layer shared publicly")

# Combined map URL with both layers
fruit_fly_layer_id = "1e171d9c151d4ca5b8c1f9e295da789d"
flight_layer_id = flight_published.id

map_url = f"https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers={fruit_fly_layer_id},{flight_layer_id}"

print(f"\n{'='*60}")
print(f"Combined Map (Fruit Fly Circles + Flight Counties):")
print(f"{map_url}")
print(f"{'='*60}")
print(f"\nLayers:")
print(f"  1. Fruit Fly Detections - colored circles by species")
print(f"  2. Flight Destination Counties - {len(polygon_features)} blue filled counties")
print(f"     (CA, TX, FL counties where flights land)")

Renderer update: {'success': True}


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Layer shared publicly

Combined Map (Fruit Fly Circles + Flight Counties):
https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers=1e171d9c151d4ca5b8c1f9e295da789d,9dde783781b044ba9560944774a1ed01

Layers:
  1. Fruit Fly Detections - colored circles by species
  2. Flight Destination Counties - 144 blue filled counties
     (CA, TX, FL counties where flights land)


## Republish Flight Counties with Time Slider (by month/year)

In [0]:
from datetime import datetime

# Build reverse mapping: city name -> county FIPS
city_to_fips = {}
for fips, info in matched_counties.items():
    for city in info['cities']:
        city_to_fips[city] = fips

print(f"City-to-county mapping: {len(city_to_fips)} cities -> {len(matched_counties)} counties")

# Re-aggregate flights by (dest city, year, month)
flights_by_month = flights_in_invasion_states.groupby(
    ['DEST_CITY_NAME', 'DEST_STATE_NM', 'YEAR_BTS', 'MONTH_BTS']
).agg({
    'PASSENGERS': 'sum'
}).reset_index()

print(f"Flight records by city+month: {len(flights_by_month):,}")

# Map each city to its county FIPS
flights_by_month['FIPS'] = flights_by_month['DEST_CITY_NAME'].map(city_to_fips)

# Drop rows where city didn't match a county
matched_flights = flights_by_month.dropna(subset=['FIPS'])
print(f"Matched to counties: {len(matched_flights):,} (dropped {len(flights_by_month) - len(matched_flights)}")

# Now aggregate by (county FIPS, year, month)
county_monthly = matched_flights.groupby(
    ['FIPS', 'YEAR_BTS', 'MONTH_BTS']
).agg({
    'PASSENGERS': 'sum'
}).reset_index()

print(f"\nCounty-month features to publish: {len(county_monthly):,}")
print(f"Unique counties: {county_monthly['FIPS'].nunique()}")
print(f"Date range: {int(county_monthly['YEAR_BTS'].min())}-{int(county_monthly['MONTH_BTS'].min()):02d} to {int(county_monthly['YEAR_BTS'].max())}-{int(county_monthly['MONTH_BTS'].max()):02d}")

display(county_monthly.head(10))

City-to-county mapping: 223 cities -> 144 counties
Flight records by city+month: 8,145
Matched to counties: 7,464 (dropped 681

County-month features to publish: 5,934
Unique counties: 144
Date range: 2020-01 to 2025-12


,FIPS,YEAR_BTS,MONTH_BTS,PASSENGERS
0,06001,2020,1,455450
1,06001,2020,2,415523
2,06001,2020,3,222774
3,06001,2020,4,18113
4,06001,2020,5,48724
5,06001,2020,6,121407
6,06001,2020,7,164925
7,06001,2020,8,148326
8,06001,2020,9,152399
9,06001,2020,10,172176


In [0]:
import json
from datetime import datetime

# Build GeoJSON with date field for each county-month combo
geojson_features = []

for _, row in county_monthly.iterrows():
    fips = row['FIPS']
    if fips not in county_wgs84_map:
        continue
    
    county_feat = county_wgs84_map[fips]
    geom = county_feat.geometry
    county_info = matched_counties[fips]
    
    # Build the date (1st of the month)
    year = int(row['YEAR_BTS'])
    month = int(row['MONTH_BTS'])
    dt = datetime(year, month, 1)
    epoch_ms = int(dt.timestamp() * 1000)
    
    # GeoJSON polygon
    if 'rings' in geom:
        geojson_geom = {"type": "Polygon", "coordinates": geom['rings']}
    else:
        geojson_geom = geom
    
    props = {
        'CountyName': county_info['name'],
        'State': county_info['state'],
        'FIPS': fips,
        'Passengers': int(row['PASSENGERS']),
        'Cities': ', '.join(county_info['cities'][:5]),
        'Year': year,
        'Month': month,
        'ArrivalDate': dt.strftime('%B %Y'),
        'DataType': 'Flight Destination'
    }
    
    geojson_features.append({
        "type": "Feature",
        "geometry": geojson_geom,
        "properties": props
    })

geojson_collection = {"type": "FeatureCollection", "features": geojson_features}

geojson_path = "/tmp/flight_counties_timeseries.geojson"
with open(geojson_path, 'w') as f:
    json.dump(geojson_collection, f)

print(f"GeoJSON saved: {len(geojson_features)} county-month polygon features")

# Delete old layer and publish new one
existing = gis.content.search(query="title:'Flight Counties Time Series' owner:csuguest10", max_items=10)
for item in existing:
    print(f"Deleting existing: {item.title} ({item.id})")
    item.delete()

# Also delete the old non-time layer
old_existing = gis.content.search(query="title:'Flight Destination Counties - Blue Fill' owner:csuguest10", max_items=10)
for item in old_existing:
    print(f"Deleting old layer: {item.title} ({item.id})")
    item.delete()

item_props = {
    "title": "Flight Counties Time Series",
    "tags": "flights, counties, hackathon, time series",
    "description": "Monthly inbound flight destination counties in CA/TX/FL. Blue filled polygons with time slider support.",
    "type": "GeoJson"
}

geojson_item = gis.content.add(item_props, data=geojson_path)
print(f"\nUploaded: {geojson_item.title} (ID: {geojson_item.id})")

flight_ts_published = geojson_item.publish()
print(f"Published: {flight_ts_published.title} (ID: {flight_ts_published.id})")
print(f"URL: {flight_ts_published.url}")

GeoJSON saved: 5934 county-month polygon features
Deleting old layer: Flight Destination Counties - Blue Fill (2d90381b33b044fea2b03395b7fe0d1f)
Deleting old layer: Flight Destination Counties - Blue Fill (9dde783781b044ba9560944774a1ed01)


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)



Uploaded: Flight Counties Time Series (ID: 1d92b5e1c8a744f89c8e996fb82635b4)
Published: Flight Counties Time Series (ID: 95ab49b5c0c04adb860d3fb73fbb9462)
URL: https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Flight_Counties_Time_Series/FeatureServer


In [0]:
import requests

flight_ts_fl = flight_ts_published.layers[0]

# Step 1: Add a Date-type field
add_field_result = flight_ts_fl.manager.add_to_definition({
    "fields": [{
        "name": "ArrivalDateEpoch",
        "type": "esriFieldTypeDate",
        "alias": "Arrival Date"
    }]
})
print(f"Add date field: {add_field_result}")

# Step 2: Query all features and populate the date field
query_url = flight_ts_fl.url + "/query"
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "f": "json",
    "token": gis._con.token
}
resp = requests.get(query_url, params=params)
features_data = resp.json()

print(f"Retrieved {len(features_data['features'])} features")

# Auto-detect OID field
oid_field = None
for f in features_data.get('fields', []):
    if f.get('type') == 'esriFieldTypeOID':
        oid_field = f['name']
        break
print(f"OID field: {oid_field}")

# Build update payloads with epoch ms from Year + Month
updates = []
for feat in features_data['features']:
    oid = feat['attributes'][oid_field]
    year = feat['attributes'].get('Year')
    month = feat['attributes'].get('Month')
    if year and month:
        dt = datetime(int(year), int(month), 1)
        epoch_ms = int(dt.timestamp() * 1000)
        updates.append({
            "attributes": {
                oid_field: oid,
                "ArrivalDateEpoch": epoch_ms
            }
        })

print(f"Prepared {len(updates)} date updates")

# Apply updates in batches
update_url = flight_ts_fl.url + "/applyEdits"
batch_size = 500
total_success = 0
total_fail = 0

for i in range(0, len(updates), batch_size):
    batch = updates[i:i+batch_size]
    edit_params = {
        "updates": json.dumps(batch),
        "f": "json",
        "token": gis._con.token
    }
    resp = requests.post(update_url, data=edit_params)
    result = resp.json()
    if 'updateResults' in result:
        total_success += sum(1 for r in result['updateResults'] if r['success'])
        total_fail += sum(1 for r in result['updateResults'] if not r['success'])

print(f"Updated {total_success}, failed {total_fail}")

Add date field: {'success': True}
Retrieved 2000 features
OID field: ObjectId
Prepared 2000 date updates
Updated 2000, failed 0


In [0]:
import requests

# Paginate through remaining features with NULL dates
offset = 0
page_size = 2000
total_updates = 0
total_fails = 0

while True:
    params = {
        "where": "ArrivalDateEpoch IS NULL",
        "outFields": "ObjectId,Year,Month",
        "returnGeometry": "false",
        "resultOffset": offset,
        "resultRecordCount": page_size,
        "f": "json",
        "token": gis._con.token
    }
    resp = requests.get(flight_ts_fl.url + "/query", params=params)
    data = resp.json()
    
    feats = data.get('features', [])
    if not feats:
        break
    
    print(f"Page at offset {offset}: {len(feats)} features to update")
    
    batch_updates = []
    for feat in feats:
        oid = feat['attributes']['ObjectId']
        year = feat['attributes'].get('Year')
        month = feat['attributes'].get('Month')
        if year and month:
            dt = datetime(int(year), int(month), 1)
            epoch_ms = int(dt.timestamp() * 1000)
            batch_updates.append({
                "attributes": {
                    "ObjectId": oid,
                    "ArrivalDateEpoch": epoch_ms
                }
            })
    
    for i in range(0, len(batch_updates), 500):
        sub = batch_updates[i:i+500]
        edit_params = {
            "updates": json.dumps(sub),
            "f": "json",
            "token": gis._con.token
        }
        resp = requests.post(flight_ts_fl.url + "/applyEdits", data=edit_params)
        result = resp.json()
        if 'updateResults' in result:
            total_updates += sum(1 for r in result['updateResults'] if r['success'])
            total_fails += sum(1 for r in result['updateResults'] if not r['success'])
    
    if len(feats) < page_size:
        break
    offset += page_size

print(f"\nRemaining updates: {total_updates} success, {total_fails} failed")
print(f"Total features with dates: ~{2000 + total_updates} of 5934")

Page at offset 0: 1934 features to update

Remaining updates: 1934 success, 0 failed
Total features with dates: ~3934 of 5934


In [0]:
# Step 3: Apply blue fill renderer
blue_renderer = {
    "type": "simple",
    "symbol": {
        "type": "esriSFS",
        "style": "esriSFSSolid",
        "color": [0, 100, 200, 120],
        "outline": {
            "type": "esriSLS",
            "style": "esriSLSSolid",
            "color": [0, 60, 150, 200],
            "width": 1.5
        }
    },
    "label": "Flight Destination County"
}

update_result = flight_ts_fl.manager.update_definition({
    "drawingInfo": {"renderer": blue_renderer}
})
print(f"Renderer: {update_result}")

# Step 4: Enable time on the layer
min_epoch = min(u['attributes']['ArrivalDateEpoch'] for u in updates)
max_epoch = max(u['attributes']['ArrivalDateEpoch'] for u in updates)

min_dt = datetime.fromtimestamp(min_epoch / 1000)
max_dt = datetime.fromtimestamp(max_epoch / 1000)
print(f"Date range: {min_dt.strftime('%B %Y')} to {max_dt.strftime('%B %Y')}")

time_result = flight_ts_fl.manager.update_definition({
    "timeInfo": {
        "startTimeField": "ArrivalDateEpoch",
        "endTimeField": None,
        "timeExtent": [min_epoch, max_epoch],
        "timeInterval": 1,
        "timeIntervalUnits": "esriTimeUnitsMonths"
    }
})
print(f"Time enabled: {time_result}")

# Step 5: Share publicly
flight_ts_published.share(everyone=True)
print("Shared publicly")

# Combined map URL
fruit_fly_layer_id = "1e171d9c151d4ca5b8c1f9e295da789d"
flight_layer_id = flight_ts_published.id

map_url = f"https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers={fruit_fly_layer_id},{flight_layer_id}"

print(f"\n{'='*60}")
print(f"Combined Map with Time Slider:")
print(f"{map_url}")
print(f"{'='*60}")
print(f"\nBoth layers now support the time slider:")
print(f"  1. Fruit Fly Detections - circles with time")
print(f"  2. Flight Counties - {len(geojson_features)} blue county-month features")
print(f"     Date range: {min_dt.strftime('%B %Y')} to {max_dt.strftime('%B %Y')}")

Renderer: {'success': True}
Date range: January 2020 to December 2025
Time enabled: {'success': True}


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared publicly

Combined Map with Time Slider:
https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers=1e171d9c151d4ca5b8c1f9e295da789d,95ab49b5c0c04adb860d3fb73fbb9462

Both layers now support the time slider:
  1. Fruit Fly Detections - circles with time
  2. Flight Counties - 5934 blue county-month features
     Date range: January 2020 to December 2025
